In [ ]:
# Random Forest Peak Risk Classifier

## Objective

Develop a Random Forest classifier to estimate the probability of an electricity
Peak Risk event for each hour within a 24-hour-ahead prediction horizon.

The model is developed independently from the electricity demand forecasting models.

### Evaluation Protocol

Expanding-window backtesting:

- Fold 2023: Train 2021–2022 → Test 2023
- Fold 2024: Train 2021–2023 → Test 2024
- Fold 2025: Train 2021–2024 → Test 2025

### Evaluation Metrics

- Precision
- Recall
- F1 Score
- Balanced Accuracy
- Positive Rate
- PR-AUC
- ROC-AUC
- Brier Score

In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

print("Imports loaded successfully.")

Imports loaded successfully.


In [3]:
# ============================================================
# 2. LOAD MASTER HOURLY DATASETS
# ============================================================

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

downtown_path = (
    PROCESSED_DATA_DIR
    / "master_hourly_downtown.csv"
)

airport_west_path = (
    PROCESSED_DATA_DIR
    / "master_hourly_airport_west.csv"
)

downtown = pd.read_csv(
    downtown_path,
    parse_dates=["TIMESTAMP"]
)

airport_west = pd.read_csv(
    airport_west_path,
    parse_dates=["TIMESTAMP"]
)

print("=" * 60)
print("DOWNTOWN")
print("=" * 60)
print("Shape:", downtown.shape)
print(
    "Period:",
    downtown["TIMESTAMP"].min(),
    "to",
    downtown["TIMESTAMP"].max()
)
print("Columns:")
print(downtown.columns.tolist())

print()

print("=" * 60)
print("AIRPORT_WEST")
print("=" * 60)
print("Shape:", airport_west.shape)
print(
    "Period:",
    airport_west["TIMESTAMP"].min(),
    "to",
    airport_west["TIMESTAMP"].max()
)
print("Columns:")
print(airport_west.columns.tolist())

DOWNTOWN
Shape: (43824, 16)
Period: 2021-01-01 00:00:00 to 2025-12-31 23:00:00
Columns:
['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'Temp (°C)', 'Rel Hum (%)', 'date', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'is_workday', 'is_public_holiday', 'holiday_name', 'is_daylight_saving_time', 'is_dst_transition_day']

AIRPORT_WEST
Shape: (43824, 16)
Period: 2021-01-01 00:00:00 to 2025-12-31 23:00:00
Columns:
['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'Temp (°C)', 'Rel Hum (%)', 'date', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'is_workday', 'is_public_holiday', 'holiday_name', 'is_daylight_saving_time', 'is_dst_transition_day']


In [4]:
# ============================================================
# 3. PEAK RISK CLASSIFICATION CONFIGURATION
# ============================================================

FORECAST_HORIZON = 24

# Expanding-window folds
FOLDS = {
    2023: {
        "train_start": "2021-01-01 00:00:00",
        "train_end":   "2022-12-31 23:00:00",
        "test_start":  "2023-01-01 00:00:00",
        "test_end":    "2023-12-31 23:00:00",
    },
    2024: {
        "train_start": "2021-01-01 00:00:00",
        "train_end":   "2023-12-31 23:00:00",
        "test_start":  "2024-01-01 00:00:00",
        "test_end":    "2024-12-31 23:00:00",
    },
    2025: {
        "train_start": "2021-01-01 00:00:00",
        "train_end":   "2024-12-31 23:00:00",
        "test_start":  "2025-01-01 00:00:00",
        "test_end":    "2025-12-31 23:00:00",
    }
}

print("Forecast horizon:", FORECAST_HORIZON, "hours")
print("Folds:", list(FOLDS.keys()))

Forecast horizon: 24 hours
Folds: [2023, 2024, 2025]


In [5]:
# ============================================================
# 4. COMBINE AND VALIDATE MASTER DATA
# ============================================================

peak_risk_data = pd.concat(
    [downtown.copy(), airport_west.copy()],
    ignore_index=True
)

peak_risk_data = (
    peak_risk_data
    .sort_values(["REGION", "TIMESTAMP"])
    .reset_index(drop=True)
)

print("=" * 60)
print("PEAK RISK MASTER DATA")
print("=" * 60)

print("Shape:", peak_risk_data.shape)

print("\nRows by region:")
print(
    peak_risk_data
    .groupby("REGION")
    .size()
)

print(
    "\nDuplicate REGION/TIMESTAMP:",
    peak_risk_data.duplicated(
        ["REGION", "TIMESTAMP"]
    ).sum()
)

print(
    "\nMissing consumption:",
    peak_risk_data["TOTAL_CONSUMPTION"].isna().sum()
)

print(
    "\nPeriod:",
    peak_risk_data["TIMESTAMP"].min(),
    "to",
    peak_risk_data["TIMESTAMP"].max()
)

PEAK RISK MASTER DATA
Shape: (87648, 16)

Rows by region:
REGION
AIRPORT_WEST    43824
DOWNTOWN        43824
dtype: int64

Duplicate REGION/TIMESTAMP: 0

Missing consumption: 0

Period: 2021-01-01 00:00:00 to 2025-12-31 23:00:00


In [6]:
# ============================================================
# 5. COMPUTE FOLD-SPECIFIC P97.5 PEAK THRESHOLDS
# ============================================================

fold_thresholds = []

for fold_year, config in FOLDS.items():

    train_mask = (
        (peak_risk_data["TIMESTAMP"] >= config["train_start"])
        &
        (peak_risk_data["TIMESTAMP"] <= config["train_end"])
    )

    train_fold = peak_risk_data.loc[train_mask]

    thresholds = (
        train_fold
        .groupby("REGION")["TOTAL_CONSUMPTION"]
        .quantile(0.975)
    )

    for region, threshold in thresholds.items():

        fold_thresholds.append(
            {
                "FOLD": fold_year,
                "REGION": region,
                "TRAIN_START": config["train_start"],
                "TRAIN_END": config["train_end"],
                "PEAK_THRESHOLD": threshold
            }
        )

fold_thresholds = pd.DataFrame(
    fold_thresholds
)

display(
    fold_thresholds.round(2)
)

,FOLD,REGION,TRAIN_START,TRAIN_END,PEAK_THRESHOLD
0,2023,AIRPORT_WEST,2021-01-01 00:00:00,2022-12-31 23:00:00,"47,444.0100"
1,2023,DOWNTOWN,2021-01-01 00:00:00,2022-12-31 23:00:00,"35,252.9500"
2,2024,AIRPORT_WEST,2021-01-01 00:00:00,2023-12-31 23:00:00,"48,865.8500"
3,2024,DOWNTOWN,2021-01-01 00:00:00,2023-12-31 23:00:00,"34,974.1200"
4,2025,AIRPORT_WEST,2021-01-01 00:00:00,2024-12-31 23:00:00,"50,884.6000"
5,2025,DOWNTOWN,2021-01-01 00:00:00,2024-12-31 23:00:00,"35,115.1900"


In [8]:
# ============================================================
# 6. BUILD 24-HOUR-AHEAD PEAK RISK DATASET
# ============================================================

def build_peak_risk_24h_dataset(df_region, max_horizon=24):

    data = (
        df_region
        .sort_values("TIMESTAMP")
        .reset_index(drop=True)
        .copy()
    )

    data = data.set_index("TIMESTAMP")

    rows = []

    timestamps = data.index

    for origin_idx in range(168, len(data) - max_horizon):

        forecast_origin = timestamps[origin_idx]

        # Historical information available at forecast origin
        load_at_origin = data.iloc[origin_idx]["TOTAL_CONSUMPTION"]

        load_lag_24 = data.iloc[
            origin_idx - 24
        ]["TOTAL_CONSUMPTION"]

        load_lag_48 = data.iloc[
            origin_idx - 48
        ]["TOTAL_CONSUMPTION"]

        load_lag_168 = data.iloc[
            origin_idx - 168
        ]["TOTAL_CONSUMPTION"]

        historical_24h = data.iloc[
            origin_idx - 23:
            origin_idx + 1
        ]["TOTAL_CONSUMPTION"]

        historical_168h = data.iloc[
            origin_idx - 167:
            origin_idx + 1
        ]["TOTAL_CONSUMPTION"]

        rolling_mean_24 = historical_24h.mean()
        rolling_max_24 = historical_24h.max()

        rolling_mean_168 = historical_168h.mean()
        rolling_max_168 = historical_168h.max()

        for horizon in range(1, max_horizon + 1):

            target_idx = origin_idx + horizon
            target_timestamp = timestamps[target_idx]

            target_row = data.iloc[target_idx]

            # Same target hour from previous day/week
            previous_day_idx = target_idx - 24
            previous_week_idx = target_idx - 168

            previous_day_load = (
                data.iloc[previous_day_idx]["TOTAL_CONSUMPTION"]
            )

            previous_week_load = (
                data.iloc[previous_week_idx]["TOTAL_CONSUMPTION"]
            )

            rows.append({
                "REGION":
                    target_row["REGION"],

                "FORECAST_ORIGIN":
                    forecast_origin,

                "TARGET_TIMESTAMP":
                    target_timestamp,

                "FORECAST_HORIZON_HOURS":
                    horizon,

                # Calendar information known in advance
                "HOUR":
                    target_row["hour"],

                "WEEKDAY":
                    target_row["weekday"],

                "MONTH":
                    target_row["month"],

                "IS_WEEKEND":
                    target_row["is_weekend"],

                "IS_WORKDAY":
                    target_row["is_workday"],

                "IS_PUBLIC_HOLIDAY":
                    target_row["is_public_holiday"],

                "IS_DAYLIGHT_SAVING_TIME":
                    target_row["is_daylight_saving_time"],

                "IS_DST_TRANSITION_DAY":
                    target_row["is_dst_transition_day"],

                # Historical demand available at origin
                "LOAD_AT_ORIGIN":
                    load_at_origin,

                "LOAD_ORIGIN_LAG_24H":
                    load_lag_24,

                "LOAD_ORIGIN_LAG_48H":
                    load_lag_48,

                "LOAD_ORIGIN_LAG_168H":
                    load_lag_168,

                "ORIGIN_ROLLING_MEAN_24H":
                    rolling_mean_24,

                "ORIGIN_ROLLING_MAX_24H":
                    rolling_max_24,

                "ORIGIN_ROLLING_MEAN_168H":
                    rolling_mean_168,

                "ORIGIN_ROLLING_MAX_168H":
                    rolling_max_168,

                "TARGET_HOUR_PREVIOUS_DAY":
                    previous_day_load,

                "TARGET_HOUR_PREVIOUS_WEEK":
                    previous_week_load,

                # Target retained only for labeling/evaluation
                "TARGET_CONSUMPTION":
                    target_row["TOTAL_CONSUMPTION"]
            })

    return pd.DataFrame(rows)

In [9]:
# ============================================================
# 6.1 BUILD DATASET FOR BOTH REGIONS
# ============================================================

downtown_peak_dataset = build_peak_risk_24h_dataset(
    downtown,
    max_horizon=24
)

airport_west_peak_dataset = build_peak_risk_24h_dataset(
    airport_west,
    max_horizon=24
)

peak_risk_24h = pd.concat(
    [
        downtown_peak_dataset,
        airport_west_peak_dataset
    ],
    ignore_index=True
)

print("Downtown:", downtown_peak_dataset.shape)
print("Airport-West:", airport_west_peak_dataset.shape)
print("Combined:", peak_risk_24h.shape)

print(
    "\nForecast horizon:",
    peak_risk_24h[
        "FORECAST_HORIZON_HOURS"
    ].min(),
    "to",
    peak_risk_24h[
        "FORECAST_HORIZON_HOURS"
    ].max()
)

print(
    "\nMissing values:",
    peak_risk_24h.isna().sum().sum()
)

Downtown: (1047168, 23)
Airport-West: (1047168, 23)
Combined: (2094336, 23)

Forecast horizon: 1 to 24

Missing values: 0


In [10]:
# ============================================================
# 7. TEMPORAL LEAKAGE AUDIT
# ============================================================

audit = peak_risk_24h.copy()

# Reconstruct timestamps represented by historical target-hour features
audit["PREVIOUS_DAY_TIMESTAMP"] = (
    audit["TARGET_TIMESTAMP"]
    - pd.Timedelta(hours=24)
)

audit["PREVIOUS_WEEK_TIMESTAMP"] = (
    audit["TARGET_TIMESTAMP"]
    - pd.Timedelta(hours=168)
)

# A historical feature is valid only if its source timestamp
# is <= FORECAST_ORIGIN
audit["PREV_DAY_SAFE"] = (
    audit["PREVIOUS_DAY_TIMESTAMP"]
    <= audit["FORECAST_ORIGIN"]
)

audit["PREV_WEEK_SAFE"] = (
    audit["PREVIOUS_WEEK_TIMESTAMP"]
    <= audit["FORECAST_ORIGIN"]
)

print("=" * 60)
print("TEMPORAL LEAKAGE AUDIT")
print("=" * 60)

print(
    "Previous-day unsafe rows:",
    (~audit["PREV_DAY_SAFE"]).sum()
)

print(
    "Previous-week unsafe rows:",
    (~audit["PREV_WEEK_SAFE"]).sum()
)

print("\nBy forecast horizon:")

display(
    audit.groupby("FORECAST_HORIZON_HOURS")
    .agg(
        ROWS=("REGION", "size"),
        PREV_DAY_SAFE=("PREV_DAY_SAFE", "mean"),
        PREV_WEEK_SAFE=("PREV_WEEK_SAFE", "mean")
    )
)

TEMPORAL LEAKAGE AUDIT
Previous-day unsafe rows: 0
Previous-week unsafe rows: 0

By forecast horizon:


,ROWS,PREV_DAY_SAFE,PREV_WEEK_SAFE
FORECAST_HORIZON_HOURS,,,
1,87264,1.0000,1.0000
2,87264,1.0000,1.0000
3,87264,1.0000,1.0000
4,87264,1.0000,1.0000
5,87264,1.0000,1.0000
6,87264,1.0000,1.0000
7,87264,1.0000,1.0000
8,87264,1.0000,1.0000
9,87264,1.0000,1.0000


In [12]:
# ============================================================
# 8. BUILD FOLD-SPECIFIC PEAK RISK TARGETS
# ============================================================

fold_datasets = {}

for fold_year, config in FOLDS.items():

    # --------------------------------------------------------
    # Select observations belonging to this fold
    # --------------------------------------------------------

    train_mask = (
        (peak_risk_24h["TARGET_TIMESTAMP"] >= config["train_start"])
        &
        (peak_risk_24h["TARGET_TIMESTAMP"] <= config["train_end"])
    )

    test_mask = (
        (peak_risk_24h["TARGET_TIMESTAMP"] >= config["test_start"])
        &
        (peak_risk_24h["TARGET_TIMESTAMP"] <= config["test_end"])
    )

    fold_data = peak_risk_24h.loc[
        train_mask | test_mask
    ].copy()

    # --------------------------------------------------------
    # Add region-specific threshold
    # --------------------------------------------------------

    thresholds_this_fold = (
        fold_thresholds[
            fold_thresholds["FOLD"] == fold_year
        ][
            ["REGION", "PEAK_THRESHOLD"]
        ]
    )

    fold_data = fold_data.merge(
        thresholds_this_fold,
        on="REGION",
        how="left",
        validate="many_to_one"
    )

    # --------------------------------------------------------
    # Create binary target
    # --------------------------------------------------------

    fold_data["ACTUAL_PEAK_RISK"] = (
        fold_data["TARGET_CONSUMPTION"]
        >= fold_data["PEAK_THRESHOLD"]
    ).astype(int)

    # --------------------------------------------------------
    # Assign dataset role
    # --------------------------------------------------------

    fold_data["DATASET_ROLE"] = np.where(
        fold_data["TARGET_TIMESTAMP"] <= config["train_end"],
        "TRAIN",
        "TEST"
    )

    fold_datasets[fold_year] = fold_data


print("Fold datasets created successfully.")

Fold datasets created successfully.


In [13]:
# ============================================================
# 9. AUDIT PEAK RISK DISTRIBUTION BY FOLD
# ============================================================

distribution_rows = []

for fold_year, fold_data in fold_datasets.items():

    summary = (
        fold_data
        .groupby(
            ["REGION", "DATASET_ROLE"]
        )["ACTUAL_PEAK_RISK"]
        .agg(
            TOTAL_ROWS="count",
            PEAK_ROWS="sum"
        )
        .reset_index()
    )

    summary["PEAK_RATE_PERCENT"] = (
        summary["PEAK_ROWS"]
        / summary["TOTAL_ROWS"]
        * 100
    )

    summary.insert(
        0,
        "FOLD",
        fold_year
    )

    distribution_rows.append(summary)


peak_distribution = pd.concat(
    distribution_rows,
    ignore_index=True
)

display(
    peak_distribution.round(2)
)

,FOLD,REGION,DATASET_ROLE,TOTAL_ROWS,PEAK_ROWS,PEAK_RATE_PERCENT
0,2023,AIRPORT_WEST,TEST,210240,8568,4.0800
1,2023,AIRPORT_WEST,TRAIN,416148,10512,2.5300
2,2023,DOWNTOWN,TEST,210240,4008,1.9100
3,2023,DOWNTOWN,TRAIN,416148,10512,2.5300
4,2024,AIRPORT_WEST,TEST,210816,11184,5.3100
5,2024,AIRPORT_WEST,TRAIN,626388,15768,2.5200
6,2024,DOWNTOWN,TEST,210816,6072,2.8800
7,2024,DOWNTOWN,TRAIN,626388,15768,2.5200
8,2025,AIRPORT_WEST,TEST,209964,12168,5.8000
9,2025,AIRPORT_WEST,TRAIN,837204,21048,2.5100


In [14]:
# ============================================================
# 10. RANDOM FOREST FEATURE SET — C1
# Calendar + Historical Demand
# ============================================================

RF_FEATURES = [

    # Forecast structure
    "FORECAST_HORIZON_HOURS",

    # Target calendar
    "HOUR",
    "WEEKDAY",
    "MONTH",
    "IS_WEEKEND",
    "IS_WORKDAY",
    "IS_PUBLIC_HOLIDAY",
    "IS_DAYLIGHT_SAVING_TIME",
    "IS_DST_TRANSITION_DAY",

    # Demand known at forecast origin
    "LOAD_AT_ORIGIN",
    "LOAD_ORIGIN_LAG_24H",
    "LOAD_ORIGIN_LAG_48H",
    "LOAD_ORIGIN_LAG_168H",

    # Historical demand summaries
    "ORIGIN_ROLLING_MEAN_24H",
    "ORIGIN_ROLLING_MAX_24H",
    "ORIGIN_ROLLING_MEAN_168H",
    "ORIGIN_ROLLING_MAX_168H",

    # Historical reference for target hour
    "TARGET_HOUR_PREVIOUS_DAY",
    "TARGET_HOUR_PREVIOUS_WEEK"
]

TARGET = "ACTUAL_PEAK_RISK"

print("Random Forest features:", len(RF_FEATURES))

for feature in RF_FEATURES:
    print("-", feature)

Random Forest features: 19
- FORECAST_HORIZON_HOURS
- HOUR
- WEEKDAY
- MONTH
- IS_WEEKEND
- IS_WORKDAY
- IS_PUBLIC_HOLIDAY
- IS_DAYLIGHT_SAVING_TIME
- IS_DST_TRANSITION_DAY
- LOAD_AT_ORIGIN
- LOAD_ORIGIN_LAG_24H
- LOAD_ORIGIN_LAG_48H
- LOAD_ORIGIN_LAG_168H
- ORIGIN_ROLLING_MEAN_24H
- ORIGIN_ROLLING_MAX_24H
- ORIGIN_ROLLING_MEAN_168H
- ORIGIN_ROLLING_MAX_168H
- TARGET_HOUR_PREVIOUS_DAY
- TARGET_HOUR_PREVIOUS_WEEK


In [15]:
# ============================================================
# 11. FEATURE INTEGRITY VALIDATION
# ============================================================

for fold_year, fold_data in fold_datasets.items():

    missing = (
        fold_data[RF_FEATURES]
        .isna()
        .sum()
        .sum()
    )

    infinite = np.isinf(
        fold_data[RF_FEATURES]
        .to_numpy(dtype=float)
    ).sum()

    print("=" * 60)
    print("FOLD:", fold_year)
    print("Rows:", len(fold_data))
    print("Features:", len(RF_FEATURES))
    print("Missing feature values:", missing)
    print("Infinite feature values:", infinite)

FOLD: 2023
Rows: 1252776
Features: 19
Missing feature values: 0
Infinite feature values: 0
FOLD: 2024
Rows: 1674408
Features: 19
Missing feature values: 0
Infinite feature values: 0
FOLD: 2025
Rows: 2094336
Features: 19
Missing feature values: 0
Infinite feature values: 0


In [16]:
# ============================================================
# 12. CLASSIFICATION EVALUATION FUNCTION
# ============================================================

def evaluate_peak_risk_classifier(
    y_true,
    y_pred,
    y_prob
):
    """
    Calculate the metrics required by the common
    Peak Risk model comparison table.
    """

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    balanced_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    # Percentage of observations classified as Peak Risk
    positive_rate = y_pred.mean() * 100

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    brier = brier_score_loss(
        y_true,
        y_prob
    )

    return {
        "PRECISION": precision,
        "RECALL": recall,
        "F1": f1,
        "BALANCED_ACCURACY": balanced_acc,
        "POSITIVE_RATE_PERCENT": positive_rate,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "BRIER": brier
    }


print("Evaluation function created successfully.")

Evaluation function created successfully.


In [17]:
# ============================================================
# 13. RANDOM FOREST V1 CONFIGURATION
# ============================================================

RF_V1_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "n_jobs": -1,
    "random_state": RANDOM_STATE
}

print("Random Forest v1 configuration")

for parameter, value in RF_V1_PARAMS.items():
    print(f"{parameter}: {value}")

Random Forest v1 configuration
n_estimators: 300
max_depth: 12
min_samples_leaf: 5
max_features: sqrt
class_weight: balanced_subsample
n_jobs: -1
random_state: 42


In [18]:
# ============================================================
# 14. RANDOM FOREST V1 — EXPANDING-WINDOW BACKTEST
# ============================================================

import time

rf_fold_results = []
rf_predictions = []
rf_models = {}

REGIONS = [
    "DOWNTOWN",
    "AIRPORT_WEST"
]

for fold_year in [2023, 2024, 2025]:

    fold_data = fold_datasets[fold_year]

    for region in REGIONS:

        print("=" * 70)
        print(
            f"RANDOM FOREST V1 | "
            f"FOLD {fold_year} | {region}"
        )
        print("=" * 70)

        region_data = (
            fold_data[
                fold_data["REGION"] == region
            ]
            .copy()
        )

        train = region_data[
            region_data["DATASET_ROLE"] == "TRAIN"
        ].copy()

        test = region_data[
            region_data["DATASET_ROLE"] == "TEST"
        ].copy()

        X_train = train[RF_FEATURES]
        y_train = train[TARGET]

        X_test = test[RF_FEATURES]
        y_test = test[TARGET]

        print("Train:", X_train.shape)
        print("Test:", X_test.shape)

        print(
            "Train peak rate:",
            f"{y_train.mean() * 100:.2f}%"
        )

        print(
            "Test peak rate:",
            f"{y_test.mean() * 100:.2f}%"
        )

        model = RandomForestClassifier(
            **RF_V1_PARAMS
        )

        start_time = time.time()

        model.fit(
            X_train,
            y_train
        )

        fit_time = time.time() - start_time

        # Probability of Peak Risk
        y_prob = model.predict_proba(
            X_test
        )[:, 1]

        # Initial decision threshold
        threshold = 0.50

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        metrics = evaluate_peak_risk_classifier(
            y_test,
            y_pred,
            y_prob
        )

        rf_fold_results.append({
            "FOLD": fold_year,
            "REGION": region,
            "THRESHOLD": threshold,
            "TRAIN_ROWS": len(train),
            "TEST_ROWS": len(test),
            "ACTUAL_POSITIVE_RATE_PERCENT":
                y_test.mean() * 100,
            "FIT_TIME_SECONDS": fit_time,
            **metrics
        })

        # Store predictions
        prediction_output = test[
            [
                "REGION",
                "FORECAST_ORIGIN",
                "TARGET_TIMESTAMP",
                "FORECAST_HORIZON_HOURS",
                TARGET
            ]
        ].copy()

        prediction_output[
            "RF_PROBABILITY"
        ] = y_prob

        prediction_output[
            "RF_PREDICTION"
        ] = y_pred

        prediction_output[
            "FOLD"
        ] = fold_year

        rf_predictions.append(
            prediction_output
        )

        rf_models[
            (fold_year, region)
        ] = model

        print(
            f"Fit time: {fit_time:.2f} seconds"
        )

        print(
            f"Precision: "
            f"{metrics['PRECISION']:.4f}"
        )

        print(
            f"Recall: "
            f"{metrics['RECALL']:.4f}"
        )

        print(
            f"F1: "
            f"{metrics['F1']:.4f}"
        )

RANDOM FOREST V1 | FOLD 2023 | DOWNTOWN
Train: (416148, 19)
Test: (210240, 19)
Train peak rate: 2.53%
Test peak rate: 1.91%
Fit time: 28.81 seconds
Precision: 0.4199
Recall: 0.5953
F1: 0.4925
RANDOM FOREST V1 | FOLD 2023 | AIRPORT_WEST
Train: (416148, 19)
Test: (210240, 19)
Train peak rate: 2.53%
Test peak rate: 4.08%
Fit time: 32.69 seconds
Precision: 0.3797
Recall: 0.8000
F1: 0.5150
RANDOM FOREST V1 | FOLD 2024 | DOWNTOWN
Train: (626388, 19)
Test: (210816, 19)
Train peak rate: 2.52%
Test peak rate: 2.88%
Fit time: 43.59 seconds
Precision: 0.4635
Recall: 0.8244
F1: 0.5934
RANDOM FOREST V1 | FOLD 2024 | AIRPORT_WEST
Train: (626388, 19)
Test: (210816, 19)
Train peak rate: 2.52%
Test peak rate: 5.31%
Fit time: 42.16 seconds
Precision: 0.4706
Recall: 0.9391
F1: 0.6270
RANDOM FOREST V1 | FOLD 2025 | DOWNTOWN
Train: (837204, 19)
Test: (209964, 19)
Train peak rate: 2.51%
Test peak rate: 8.32%
Fit time: 63.65 seconds
Precision: 0.6651
Recall: 0.8363
F1: 0.7409
RANDOM FOREST V1 | FOLD 2025 | A

In [19]:
# ============================================================
# 15. RANDOM FOREST V1 — REGIONAL BACKTEST RESULTS
# ============================================================

rf_regional_results = pd.DataFrame(
    rf_fold_results
)

display(
    rf_regional_results[
        [
            "FOLD",
            "REGION",
            "TRAIN_ROWS",
            "TEST_ROWS",
            "ACTUAL_POSITIVE_RATE_PERCENT",
            "PRECISION",
            "RECALL",
            "F1",
            "BALANCED_ACCURACY",
            "POSITIVE_RATE_PERCENT",
            "PR_AUC",
            "ROC_AUC",
            "BRIER",
            "FIT_TIME_SECONDS"
        ]
    ].round(4)
)

,FOLD,REGION,TRAIN_ROWS,TEST_ROWS,ACTUAL_POSITIVE_RATE_PERCENT,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER,FIT_TIME_SECONDS
0,2023,DOWNTOWN,416148,210240,1.9064,0.4199,0.5953,0.4925,0.7897,2.7026,0.5174,0.9783,0.0165,28.8091
1,2023,AIRPORT_WEST,416148,210240,4.0753,0.3797,0.8000,0.5150,0.8722,8.5864,0.4409,0.9663,0.0433,32.6885
2,2024,DOWNTOWN,626388,210816,2.8802,0.4635,0.8244,0.5934,0.8981,5.1234,0.6426,0.9809,0.0239,43.5918
3,2024,AIRPORT_WEST,626388,210816,5.3051,0.4706,0.9391,0.6270,0.9400,10.5870,0.6818,0.9793,0.0389,42.1557
4,2025,DOWNTOWN,837204,209964,8.3219,0.6651,0.8363,0.7409,0.8990,10.4642,0.7165,0.9732,0.0380,63.6461
5,2025,AIRPORT_WEST,837204,209964,5.7953,0.6569,0.8277,0.7325,0.9005,7.3013,0.8422,0.9892,0.0227,60.9297


In [20]:
# ============================================================
# 16. CONSOLIDATE RANDOM FOREST PREDICTIONS
# ============================================================

rf_all_predictions = pd.concat(
    rf_predictions,
    ignore_index=True
)

print("=" * 60)
print("RANDOM FOREST V1 — PREDICTION DATASET")
print("=" * 60)

print("Rows:", len(rf_all_predictions))

print("\nRows by fold:")
print(
    rf_all_predictions
    .groupby("FOLD")
    .size()
)

print("\nRows by fold and region:")
print(
    rf_all_predictions
    .groupby(["FOLD", "REGION"])
    .size()
)

print(
    "\nMissing probabilities:",
    rf_all_predictions["RF_PROBABILITY"].isna().sum()
)

RANDOM FOREST V1 — PREDICTION DATASET
Rows: 1262040

Rows by fold:
FOLD
2023    420480
2024    421632
2025    419928
dtype: int64

Rows by fold and region:
FOLD  REGION      
2023  AIRPORT_WEST    210240
      DOWNTOWN        210240
2024  AIRPORT_WEST    210816
      DOWNTOWN        210816
2025  AIRPORT_WEST    209964
      DOWNTOWN        209964
dtype: int64

Missing probabilities: 0


In [21]:
# ============================================================
# 17. RANDOM FOREST V1 — CONSOLIDATED FOLD RESULTS
# ============================================================

rf_consolidated_results = []

for fold_year in [2023, 2024, 2025]:

    fold_predictions = (
        rf_all_predictions[
            rf_all_predictions["FOLD"] == fold_year
        ]
    )

    y_true = fold_predictions[TARGET]
    y_pred = fold_predictions["RF_PREDICTION"]
    y_prob = fold_predictions["RF_PROBABILITY"]

    metrics = evaluate_peak_risk_classifier(
        y_true,
        y_pred,
        y_prob
    )

    rf_consolidated_results.append({
        "FOLD": fold_year,
        "TEST_ROWS": len(fold_predictions),
        "ACTUAL_POSITIVE_RATE_PERCENT":
            y_true.mean() * 100,
        **metrics
    })


rf_consolidated_results = pd.DataFrame(
    rf_consolidated_results
)

display(
    rf_consolidated_results.round(4)
)

,FOLD,TEST_ROWS,ACTUAL_POSITIVE_RATE_PERCENT,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER
0,2023,420480,2.9909,0.3893,0.7347,0.5090,0.8496,5.6445,0.4604,0.9735,0.0299
1,2024,421632,4.0927,0.4683,0.8988,0.6157,0.9276,7.8552,0.6701,0.9812,0.0314
2,2025,419928,7.0586,0.6618,0.8328,0.7375,0.9002,8.8827,0.7692,0.9814,0.0303


In [22]:
# ============================================================
# 18. RANDOM FOREST V1 — GLOBAL OUT-OF-SAMPLE RESULTS
# ============================================================

global_y_true = rf_all_predictions[TARGET]
global_y_pred = rf_all_predictions["RF_PREDICTION"]
global_y_prob = rf_all_predictions["RF_PROBABILITY"]

global_metrics = evaluate_peak_risk_classifier(
    global_y_true,
    global_y_pred,
    global_y_prob
)

rf_global_results = pd.DataFrame([{
    "FOLD": "GLOBAL",
    "TEST_ROWS": len(rf_all_predictions),
    "ACTUAL_POSITIVE_RATE_PERCENT":
        global_y_true.mean() * 100,
    **global_metrics
}])

display(
    rf_global_results.round(4)
)

,FOLD,TEST_ROWS,ACTUAL_POSITIVE_RATE_PERCENT,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER
0,GLOBAL,1262040,4.7124,0.5250,0.8312,0.6435,0.8970,7.4605,0.6547,0.9789,0.0305


In [23]:
# ============================================================
# 19. RANDOM FOREST V1 — MODEL COMPARISON FORMAT
# ============================================================

rf_v1_summary = pd.concat(
    [
        rf_global_results,
        rf_consolidated_results
    ],
    ignore_index=True
)

rf_v1_summary.insert(
    0,
    "MODEL",
    "Random Forest v1"
)

rf_v1_summary = rf_v1_summary[
    [
        "MODEL",
        "FOLD",
        "PRECISION",
        "RECALL",
        "F1",
        "BALANCED_ACCURACY",
        "POSITIVE_RATE_PERCENT",
        "PR_AUC",
        "ROC_AUC",
        "BRIER"
    ]
]

display(
    rf_v1_summary.round(4)
)

,MODEL,FOLD,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER
0,Random Forest v1,GLOBAL,0.5250,0.8312,0.6435,0.8970,7.4605,0.6547,0.9789,0.0305
1,Random Forest v1,2023,0.3893,0.7347,0.5090,0.8496,5.6445,0.4604,0.9735,0.0299
2,Random Forest v1,2024,0.4683,0.8988,0.6157,0.9276,7.8552,0.6701,0.9812,0.0314
3,Random Forest v1,2025,0.6618,0.8328,0.7375,0.9002,8.8827,0.7692,0.9814,0.0303


In [24]:
# ============================================================
# 20. INTERNAL TUNING SPLIT — 2021 TRAIN / 2022 VALIDATION
# ============================================================

tuning_data = peak_risk_24h[
    peak_risk_24h["TARGET_TIMESTAMP"].dt.year.isin([2021, 2022])
].copy()

# Threshold calculated ONLY from 2021
thresholds_2021 = (
    peak_risk_data[
        peak_risk_data["TIMESTAMP"].dt.year == 2021
    ]
    .groupby("REGION")["TOTAL_CONSUMPTION"]
    .quantile(0.975)
    .rename("TUNING_PEAK_THRESHOLD")
    .reset_index()
)

tuning_data = tuning_data.merge(
    thresholds_2021,
    on="REGION",
    how="left",
    validate="many_to_one"
)

tuning_data["ACTUAL_PEAK_RISK"] = (
    tuning_data["TARGET_CONSUMPTION"]
    >= tuning_data["TUNING_PEAK_THRESHOLD"]
).astype(int)

tuning_data["TUNING_ROLE"] = np.where(
    tuning_data["TARGET_TIMESTAMP"].dt.year == 2021,
    "TRAIN",
    "VALIDATION"
)

print("=" * 60)
print("INTERNAL RANDOM FOREST TUNING DATASET")
print("=" * 60)

print("\nThresholds based only on 2021:")
display(thresholds_2021.round(2))

tuning_distribution = (
    tuning_data
    .groupby(["REGION", "TUNING_ROLE"])["ACTUAL_PEAK_RISK"]
    .agg(
        TOTAL_ROWS="count",
        PEAK_ROWS="sum"
    )
    .reset_index()
)

tuning_distribution["PEAK_RATE_PERCENT"] = (
    tuning_distribution["PEAK_ROWS"]
    / tuning_distribution["TOTAL_ROWS"]
    * 100
)

display(tuning_distribution.round(2))

INTERNAL RANDOM FOREST TUNING DATASET

Thresholds based only on 2021:


,REGION,TUNING_PEAK_THRESHOLD
0,AIRPORT_WEST,"48,699.7500"
1,DOWNTOWN,"35,383.9700"


,REGION,TUNING_ROLE,TOTAL_ROWS,PEAK_ROWS,PEAK_RATE_PERCENT
0,AIRPORT_WEST,TRAIN,205908,5256,2.5500
1,AIRPORT_WEST,VALIDATION,210240,3624,1.7200
2,DOWNTOWN,TRAIN,205908,5256,2.5500
3,DOWNTOWN,VALIDATION,210240,4728,2.2500


In [25]:
# ============================================================
# 21. RANDOM FOREST TUNING CONFIGURATIONS
# ============================================================

RF_TUNING_CONFIGS = []

config_id = 1

for max_depth in [8, 12, 16]:

    for min_samples_leaf in [3, 10]:

        for class_weight in [
            "balanced",
            "balanced_subsample"
        ]:

            RF_TUNING_CONFIGS.append({
                "CONFIG_ID": config_id,
                "n_estimators": 300,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": "sqrt",
                "class_weight": class_weight,
                "n_jobs": -1,
                "random_state": RANDOM_STATE
            })

            config_id += 1


print(
    "Configurations:",
    len(RF_TUNING_CONFIGS)
)

pd.DataFrame(
    RF_TUNING_CONFIGS
)

Configurations: 12


,CONFIG_ID,n_estimators,max_depth,min_samples_leaf,max_features,class_weight,n_jobs,random_state
0,1,300,8,3,sqrt,balanced,-1,42
1,2,300,8,3,sqrt,balanced_subsample,-1,42
2,3,300,8,10,sqrt,balanced,-1,42
3,4,300,8,10,sqrt,balanced_subsample,-1,42
4,5,300,12,3,sqrt,balanced,-1,42
5,6,300,12,3,sqrt,balanced_subsample,-1,42
6,7,300,12,10,sqrt,balanced,-1,42
7,8,300,12,10,sqrt,balanced_subsample,-1,42
8,9,300,16,3,sqrt,balanced,-1,42
9,10,300,16,3,sqrt,balanced_subsample,-1,42


In [26]:
# ============================================================
# 22. RANDOM FOREST — INTERNAL HYPERPARAMETER TUNING
# ============================================================

rf_tuning_results = []

for region in REGIONS:

    print("\n" + "=" * 70)
    print("TUNING REGION:", region)
    print("=" * 70)

    region_data = tuning_data[
        tuning_data["REGION"] == region
    ]

    train = region_data[
        region_data["TUNING_ROLE"] == "TRAIN"
    ]

    validation = region_data[
        region_data["TUNING_ROLE"] == "VALIDATION"
    ]

    X_train = train[RF_FEATURES]
    y_train = train[TARGET]

    X_validation = validation[RF_FEATURES]
    y_validation = validation[TARGET]

    print("Train:", X_train.shape)
    print("Validation:", X_validation.shape)

    for params in RF_TUNING_CONFIGS:

        model_params = {
            key: value
            for key, value in params.items()
            if key != "CONFIG_ID"
        }

        start = time.time()

        model = RandomForestClassifier(
            **model_params
        )

        model.fit(
            X_train,
            y_train
        )

        y_prob = model.predict_proba(
            X_validation
        )[:, 1]

        y_pred = (
            y_prob >= 0.50
        ).astype(int)

        metrics = evaluate_peak_risk_classifier(
            y_validation,
            y_pred,
            y_prob
        )

        elapsed = time.time() - start

        rf_tuning_results.append({
            "REGION": region,
            "CONFIG_ID": params["CONFIG_ID"],
            "N_ESTIMATORS":
                params["n_estimators"],
            "MAX_DEPTH":
                params["max_depth"],
            "MIN_SAMPLES_LEAF":
                params["min_samples_leaf"],
            "MAX_FEATURES":
                params["max_features"],
            "CLASS_WEIGHT":
                params["class_weight"],
            "THRESHOLD": 0.50,
            **metrics,
            "FIT_TIME_SECONDS": elapsed
        })

        print(
            f"Config {params['CONFIG_ID']:02d} | "
            f"PR-AUC={metrics['PR_AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Recall={metrics['RECALL']:.4f}"
        )


TUNING REGION: DOWNTOWN
Train: (205908, 19)
Validation: (210240, 19)
Config 01 | PR-AUC=0.5424 | F1=0.5168 | Recall=0.7953
Config 02 | PR-AUC=0.5205 | F1=0.5226 | Recall=0.7923
Config 03 | PR-AUC=0.5381 | F1=0.5101 | Recall=0.7965
Config 04 | PR-AUC=0.5352 | F1=0.5089 | Recall=0.7887
Config 05 | PR-AUC=0.5081 | F1=0.4756 | Recall=0.4841
Config 06 | PR-AUC=0.5094 | F1=0.4683 | Recall=0.4602
Config 07 | PR-AUC=0.5155 | F1=0.4788 | Recall=0.5017
Config 08 | PR-AUC=0.5141 | F1=0.4752 | Recall=0.4833
Config 09 | PR-AUC=0.5224 | F1=0.4814 | Recall=0.4175
Config 10 | PR-AUC=0.4937 | F1=0.4756 | Recall=0.4036
Config 11 | PR-AUC=0.5248 | F1=0.4932 | Recall=0.4459
Config 12 | PR-AUC=0.5260 | F1=0.4885 | Recall=0.4344

TUNING REGION: AIRPORT_WEST
Train: (205908, 19)
Validation: (210240, 19)
Config 01 | PR-AUC=0.2983 | F1=0.4006 | Recall=0.7875
Config 02 | PR-AUC=0.2838 | F1=0.4018 | Recall=0.7864
Config 03 | PR-AUC=0.2872 | F1=0.3941 | Recall=0.7870
Config 04 | PR-AUC=0.2951 | F1=0.4004 | Recall

In [27]:
# ============================================================
# 23. RANDOM FOREST TUNING RANKING
# ============================================================

rf_tuning_results_df = pd.DataFrame(
    rf_tuning_results
)

rf_tuning_ranking = (
    rf_tuning_results_df
    .sort_values(
        [
            "REGION",
            "PR_AUC",
            "F1",
            "BALANCED_ACCURACY"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
)

for region in REGIONS:

    print("\n" + "=" * 70)
    print(region)
    print("=" * 70)

    display(
        rf_tuning_ranking[
            rf_tuning_ranking["REGION"] == region
        ][
            [
                "CONFIG_ID",
                "MAX_DEPTH",
                "MIN_SAMPLES_LEAF",
                "CLASS_WEIGHT",
                "PRECISION",
                "RECALL",
                "F1",
                "BALANCED_ACCURACY",
                "PR_AUC",
                "ROC_AUC",
                "BRIER"
            ]
        ]
        .head(5)
        .round(4)
    )


DOWNTOWN


,CONFIG_ID,MAX_DEPTH,MIN_SAMPLES_LEAF,CLASS_WEIGHT,PRECISION,RECALL,F1,BALANCED_ACCURACY,PR_AUC,ROC_AUC,BRIER
0,1,8,3,balanced,0.3828,0.7953,0.5168,0.8829,0.5424,0.9700,0.0243
2,3,8,10,balanced,0.3752,0.7965,0.5101,0.8830,0.5381,0.9703,0.0244
3,4,8,10,balanced_subsample,0.3756,0.7887,0.5089,0.8793,0.5352,0.9685,0.0246
11,12,16,10,balanced_subsample,0.5578,0.4344,0.4885,0.7133,0.5260,0.9602,0.0149
10,11,16,10,balanced,0.5517,0.4459,0.4932,0.7188,0.5248,0.9590,0.0152



AIRPORT_WEST


,CONFIG_ID,MAX_DEPTH,MIN_SAMPLES_LEAF,CLASS_WEIGHT,PRECISION,RECALL,F1,BALANCED_ACCURACY,PR_AUC,ROC_AUC,BRIER
12,1,8,3,balanced,0.2686,0.7875,0.4006,0.8750,0.2983,0.9679,0.0291
15,4,8,10,balanced_subsample,0.2686,0.7864,0.4004,0.8744,0.2951,0.9542,0.0299
14,3,8,10,balanced,0.2629,0.7870,0.3941,0.8741,0.2872,0.9656,0.0300
13,2,8,3,balanced_subsample,0.2698,0.7864,0.4018,0.8745,0.2838,0.9533,0.0293
19,8,12,10,balanced_subsample,0.2512,0.5077,0.3361,0.7406,0.2718,0.9551,0.0215


In [28]:
# ============================================================
# 24. FIT SELECTED RF CONFIGURATION FOR THRESHOLD ANALYSIS
# ============================================================

RF_FINAL_PARAMS = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 3,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "n_jobs": -1,
    "random_state": RANDOM_STATE
}

rf_threshold_models = {}
rf_threshold_validation = {}

for region in REGIONS:

    region_data = tuning_data[
        tuning_data["REGION"] == region
    ]

    train = region_data[
        region_data["TUNING_ROLE"] == "TRAIN"
    ]

    validation = region_data[
        region_data["TUNING_ROLE"] == "VALIDATION"
    ]

    X_train = train[RF_FEATURES]
    y_train = train[TARGET]

    X_validation = validation[RF_FEATURES]
    y_validation = validation[TARGET]

    model = RandomForestClassifier(
        **RF_FINAL_PARAMS
    )

    model.fit(
        X_train,
        y_train
    )

    y_prob = model.predict_proba(
        X_validation
    )[:, 1]

    rf_threshold_models[region] = model

    rf_threshold_validation[region] = {
        "y_true": y_validation.to_numpy(),
        "y_prob": y_prob
    }

    print("=" * 60)
    print(region)
    print("Train rows:", len(X_train))
    print("Validation rows:", len(X_validation))
    print(
        "Validation peak rate:",
        f"{y_validation.mean()*100:.2f}%"
    )

DOWNTOWN
Train rows: 205908
Validation rows: 210240
Validation peak rate: 2.25%
AIRPORT_WEST
Train rows: 205908
Validation rows: 210240
Validation peak rate: 1.72%


In [30]:
# ============================================================
# 25. RANDOM FOREST — THRESHOLD SWEEP ON 2022 VALIDATION
# ============================================================

from sklearn.metrics import confusion_matrix

threshold_results = []

# Exactly 61 thresholds:
# 0.20, 0.21, ..., 0.79, 0.80
threshold_grid = np.linspace(
    0.20,
    0.80,
    61
)

# Validation check
print("Number of thresholds:", len(threshold_grid))
print("Minimum threshold:", threshold_grid.min())
print("Maximum threshold:", threshold_grid.max())

for region in REGIONS:

    y_true = rf_threshold_validation[region]["y_true"]
    y_prob = rf_threshold_validation[region]["y_prob"]

    for threshold in threshold_grid:

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        metrics = evaluate_peak_risk_classifier(
            y_true,
            y_pred,
            y_prob
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred
        ).ravel()

        threshold_results.append({
            "REGION": region,
            "THRESHOLD": threshold,
            "PRECISION": metrics["PRECISION"],
            "RECALL": metrics["RECALL"],
            "F1": metrics["F1"],
            "BALANCED_ACCURACY":
                metrics["BALANCED_ACCURACY"],
            "PR_AUC": metrics["PR_AUC"],
            "ROC_AUC": metrics["ROC_AUC"],
            "TRUE_NEGATIVES": tn,
            "FALSE_POSITIVES": fp,
            "FALSE_NEGATIVES": fn,
            "TRUE_POSITIVES": tp
        })


rf_threshold_results = pd.DataFrame(
    threshold_results
)

print("\n" + "=" * 60)
print("RANDOM FOREST THRESHOLD SWEEP")
print("=" * 60)

print(
    "Threshold configurations evaluated:",
    len(rf_threshold_results)
)

print(
    "Regions:",
    rf_threshold_results["REGION"].unique()
)

print(
    "Threshold range:",
    f"{rf_threshold_results['THRESHOLD'].min():.2f}",
    "to",
    f"{rf_threshold_results['THRESHOLD'].max():.2f}"
)

Number of thresholds: 61
Minimum threshold: 0.2
Maximum threshold: 0.8

RANDOM FOREST THRESHOLD SWEEP
Threshold configurations evaluated: 122
Regions: ['DOWNTOWN' 'AIRPORT_WEST']
Threshold range: 0.20 to 0.80


In [31]:
# ============================================================
# 26. RANDOM FOREST — THRESHOLD CANDIDATES
# ============================================================

for region in REGIONS:

    region_results = (
        rf_threshold_results[
            rf_threshold_results["REGION"] == region
        ]
        .sort_values(
            ["F1", "RECALL"],
            ascending=[False, False]
        )
    )

    print("\n" + "=" * 70)
    print(region)
    print("=" * 70)

    display(
        region_results[
            [
                "THRESHOLD",
                "PRECISION",
                "RECALL",
                "F1",
                "BALANCED_ACCURACY",
                "FALSE_POSITIVES",
                "FALSE_NEGATIVES",
                "TRUE_POSITIVES"
            ]
        ]
        .head(10)
        .round(4)
    )


DOWNTOWN


,THRESHOLD,PRECISION,RECALL,F1,BALANCED_ACCURACY,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES
39,0.5900,0.4349,0.7301,0.5451,0.8541,4486,1276,3452
40,0.6000,0.4367,0.7200,0.5437,0.8493,4390,1324,3404
38,0.5800,0.4295,0.7365,0.5425,0.8570,4626,1246,3482
37,0.5700,0.4260,0.7453,0.5422,0.8611,4748,1204,3524
41,0.6100,0.4397,0.7030,0.5411,0.8412,4235,1404,3324
36,0.5600,0.4208,0.7538,0.5401,0.8650,4905,1164,3564
42,0.6200,0.4436,0.6882,0.5395,0.8342,4081,1474,3254
43,0.6300,0.4509,0.6709,0.5393,0.8260,3863,1556,3172
35,0.5500,0.4160,0.7640,0.5387,0.8696,5070,1116,3612
44,0.6400,0.4560,0.6489,0.5356,0.8155,3660,1660,3068



AIRPORT_WEST


,THRESHOLD,PRECISION,RECALL,F1,BALANCED_ACCURACY,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES
114,0.7300,0.3187,0.7127,0.4405,0.8430,5521,1041,2583
113,0.7200,0.3150,0.7285,0.4398,0.8503,5741,984,2640
115,0.7400,0.3211,0.6896,0.4382,0.8320,5284,1125,2499
116,0.7500,0.3226,0.6656,0.4346,0.8205,5065,1212,2412
112,0.7100,0.3076,0.7334,0.4334,0.8522,5983,966,2658
111,0.7000,0.3015,0.7373,0.4280,0.8537,6189,952,2672
117,0.7600,0.3213,0.6319,0.4260,0.8042,4838,1334,2290
110,0.6900,0.2973,0.7401,0.4242,0.8547,6339,942,2682
109,0.6800,0.2934,0.7431,0.4207,0.8559,6485,931,2693
108,0.6700,0.2915,0.7478,0.4195,0.8580,6587,914,2710


In [32]:
# ============================================================
# 27. THRESHOLD TRADE-OFF COMPARISON
# ============================================================

thresholds_to_compare = {
    "DOWNTOWN": [0.50, 0.55, 0.57, 0.59],
    "AIRPORT_WEST": [0.50, 0.67, 0.70, 0.72, 0.73]
}

comparison_rows = []

for region, thresholds in thresholds_to_compare.items():

    region_results = rf_threshold_results[
        rf_threshold_results["REGION"] == region
    ]

    for threshold in thresholds:

        row = region_results[
            np.isclose(
                region_results["THRESHOLD"],
                threshold
            )
        ].iloc[0]

        comparison_rows.append(row)

rf_threshold_comparison = pd.DataFrame(
    comparison_rows
)

display(
    rf_threshold_comparison[
        [
            "REGION",
            "THRESHOLD",
            "PRECISION",
            "RECALL",
            "F1",
            "BALANCED_ACCURACY",
            "FALSE_POSITIVES",
            "FALSE_NEGATIVES",
            "TRUE_POSITIVES"
        ]
    ].round(4)
)

,REGION,THRESHOLD,PRECISION,RECALL,F1,BALANCED_ACCURACY,FALSE_POSITIVES,FALSE_NEGATIVES,TRUE_POSITIVES
30,DOWNTOWN,0.5000,0.3828,0.7953,0.5168,0.8829,6062,968,3760
35,DOWNTOWN,0.5500,0.4160,0.7640,0.5387,0.8696,5070,1116,3612
37,DOWNTOWN,0.5700,0.4260,0.7453,0.5422,0.8611,4748,1204,3524
39,DOWNTOWN,0.5900,0.4349,0.7301,0.5451,0.8541,4486,1276,3452
91,AIRPORT_WEST,0.5000,0.2686,0.7875,0.4006,0.8750,7771,770,2854
108,AIRPORT_WEST,0.6700,0.2915,0.7478,0.4195,0.8580,6587,914,2710
111,AIRPORT_WEST,0.7000,0.3015,0.7373,0.4280,0.8537,6189,952,2672
113,AIRPORT_WEST,0.7200,0.3150,0.7285,0.4398,0.8503,5741,984,2640
114,AIRPORT_WEST,0.7300,0.3187,0.7127,0.4405,0.8430,5521,1041,2583


In [33]:
RF_FINAL_PARAMS = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 3,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "n_jobs": -1,
    "random_state": 42
}

RF_FINAL_THRESHOLDS = {
    "DOWNTOWN": 0.50,
    "AIRPORT_WEST": 0.50
}

### Random Forest Final Configuration

Hyperparameters were selected using the 2021 training period and
2022 validation period, keeping the 2023–2025 evaluation folds
isolated from model selection.

The selected Random Forest configuration was:

- n_estimators = 300
- max_depth = 8
- min_samples_leaf = 3
- max_features = sqrt
- class_weight = balanced

A probability threshold analysis was also performed on the 2022
validation period. Although higher thresholds improved F1 and
precision, they produced a meaningful increase in false negatives
and reduced recall.

Because the Peak Risk Classifier is intended as an early-warning
model, retaining sensitivity to actual peak-risk events was
prioritized over reducing false-positive alerts.

Therefore, the final decision threshold was kept at 0.50 for both
regions.

The final configuration and thresholds were frozen before evaluation
on the 2023, 2024, and 2025 out-of-sample folds.

In [37]:
# ============================================================
# 28. FINAL RANDOM FOREST — EXPANDING-WINDOW EVALUATION
# ============================================================

import time
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier


# ------------------------------------------------------------
# Frozen configuration selected using 2021 -> 2022 validation
# ------------------------------------------------------------

RF_FINAL_PARAMS = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 3,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "n_jobs": -1,
    "random_state": RANDOM_STATE
}

RF_FINAL_THRESHOLDS = {
    "DOWNTOWN": 0.50,
    "AIRPORT_WEST": 0.50
}


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

rf_final_results = []
rf_final_predictions = []


# ------------------------------------------------------------
# Expanding-window evaluation
# ------------------------------------------------------------

for fold_year in [2023, 2024, 2025]:

    print("\n" + "#" * 75)
    print(f"FINAL RANDOM FOREST | FOLD {fold_year}")
    print("#" * 75)

    # Correct existing dictionary created in Section 8
    fold_data = fold_datasets[fold_year]

    for region in REGIONS:

        print("\n" + "=" * 70)
        print(region)
        print("=" * 70)

        region_data = (
            fold_data[
                fold_data["REGION"] == region
            ]
            .copy()
        )

        # ----------------------------------------------------
        # Train / Test
        # ----------------------------------------------------

        train = (
            region_data[
                region_data["DATASET_ROLE"] == "TRAIN"
            ]
            .copy()
        )

        test = (
            region_data[
                region_data["DATASET_ROLE"] == "TEST"
            ]
            .copy()
        )

        X_train = train[RF_FEATURES]
        y_train = train[TARGET]

        X_test = test[RF_FEATURES]
        y_test = test[TARGET]

        threshold = RF_FINAL_THRESHOLDS[region]

        print("Train:", X_train.shape)
        print("Test:", X_test.shape)

        print(
            "Train peak rate:",
            f"{y_train.mean() * 100:.2f}%"
        )

        print(
            "Test peak rate:",
            f"{y_test.mean() * 100:.2f}%"
        )

        print(
            "Decision threshold:",
            threshold
        )

        # ----------------------------------------------------
        # Fit final frozen model
        # ----------------------------------------------------

        start_time = time.time()

        model = RandomForestClassifier(
            **RF_FINAL_PARAMS
        )

        model.fit(
            X_train,
            y_train
        )

        fit_time = time.time() - start_time

        # ----------------------------------------------------
        # Out-of-sample probabilities
        # ----------------------------------------------------

        y_prob = model.predict_proba(
            X_test
        )[:, 1]

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        # ----------------------------------------------------
        # Evaluation metrics
        # ----------------------------------------------------

        metrics = evaluate_peak_risk_classifier(
            y_test,
            y_pred,
            y_prob
        )

        rf_final_results.append({
            "FOLD": fold_year,
            "REGION": region,
            "TRAIN_ROWS": len(train),
            "TEST_ROWS": len(test),
            "ACTUAL_POSITIVE_RATE_PERCENT":
                y_test.mean() * 100,
            "THRESHOLD": threshold,
            **metrics,
            "FIT_TIME_SECONDS": fit_time
        })

        # ----------------------------------------------------
        # Preserve out-of-sample predictions
        # ----------------------------------------------------

        prediction_output = (
            test[
                [
                    "REGION",
                    "FORECAST_ORIGIN",
                    "TARGET_TIMESTAMP",
                    "FORECAST_HORIZON_HOURS"
                ]
            ]
            .copy()
        )

        prediction_output["FOLD"] = fold_year

        prediction_output[
            "ACTUAL_PEAK_RISK"
        ] = y_test.to_numpy()

        prediction_output[
            "RF_PROBABILITY"
        ] = y_prob

        prediction_output[
            "RF_PREDICTION"
        ] = y_pred

        rf_final_predictions.append(
            prediction_output
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        print(
            "Fit time:",
            f"{fit_time:.2f} seconds"
        )

        print(
            "Precision:",
            f"{metrics['PRECISION']:.4f}"
        )

        print(
            "Recall:",
            f"{metrics['RECALL']:.4f}"
        )

        print(
            "F1:",
            f"{metrics['F1']:.4f}"
        )

        print(
            "Balanced Accuracy:",
            f"{metrics['BALANCED_ACCURACY']:.4f}"
        )

        print(
            "PR-AUC:",
            f"{metrics['PR_AUC']:.4f}"
        )

        print(
            "ROC-AUC:",
            f"{metrics['ROC_AUC']:.4f}"
        )

        print(
            "Brier:",
            f"{metrics['BRIER']:.4f}"
        )


# ------------------------------------------------------------
# Consolidate results
# ------------------------------------------------------------

rf_final_results_df = pd.DataFrame(
    rf_final_results
)

rf_final_predictions_df = pd.concat(
    rf_final_predictions,
    ignore_index=True
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL RANDOM FOREST — EVALUATION COMPLETE")
print("=" * 75)

print(
    "Total out-of-sample predictions:",
    len(rf_final_predictions_df)
)

print(
    "Missing probabilities:",
    rf_final_predictions_df[
        "RF_PROBABILITY"
    ].isna().sum()
)

print(
    "Missing predictions:",
    rf_final_predictions_df[
        "RF_PREDICTION"
    ].isna().sum()
)

print(
    "Evaluation rows:",
    len(rf_final_results_df)
)

display(
    rf_final_results_df.round(4)
)


###########################################################################
FINAL RANDOM FOREST | FOLD 2023
###########################################################################

DOWNTOWN
Train: (416148, 19)
Test: (210240, 19)
Train peak rate: 2.53%
Test peak rate: 1.91%
Decision threshold: 0.5
Fit time: 19.04 seconds
Precision: 0.3308
Recall: 0.8533
F1: 0.4768
Balanced Accuracy: 0.9099
PR-AUC: 0.5099
ROC-AUC: 0.9788
Brier: 0.0263

AIRPORT_WEST
Train: (416148, 19)
Test: (210240, 19)
Train peak rate: 2.53%
Test peak rate: 4.08%
Decision threshold: 0.5
Fit time: 18.94 seconds
Precision: 0.3488
Recall: 0.8997
F1: 0.5027
Balanced Accuracy: 0.9142
PR-AUC: 0.4550
ROC-AUC: 0.9669
Brier: 0.0577

###########################################################################
FINAL RANDOM FOREST | FOLD 2024
###########################################################################

DOWNTOWN
Train: (626388, 19)
Test: (210816, 19)
Train peak rate: 2.52%
Test peak rate: 2.88%
Decision threshold

,FOLD,REGION,TRAIN_ROWS,TEST_ROWS,ACTUAL_POSITIVE_RATE_PERCENT,THRESHOLD,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER,FIT_TIME_SECONDS
0,2023,DOWNTOWN,416148,210240,1.9064,0.5000,0.3308,0.8533,0.4768,0.9099,4.9172,0.5099,0.9788,0.0263,19.0420
1,2023,AIRPORT_WEST,416148,210240,4.0753,0.5000,0.3488,0.8997,0.5027,0.9142,10.5132,0.4550,0.9669,0.0577,18.9436
2,2024,DOWNTOWN,626388,210816,2.8802,0.5000,0.3219,0.9293,0.4781,0.9356,8.3163,0.6593,0.9797,0.0409,39.6112
3,2024,AIRPORT_WEST,626388,210816,5.3051,0.5000,0.4135,0.9664,0.5791,0.9448,12.3994,0.7038,0.9806,0.0551,41.9431
4,2025,DOWNTOWN,837204,209964,8.3219,0.5000,0.4893,0.9146,0.6375,0.9140,15.5536,0.7356,0.9719,0.0589,79.4928
5,2025,AIRPORT_WEST,837204,209964,5.7953,0.5000,0.5697,0.9736,0.7188,0.9642,9.9041,0.8499,0.9894,0.0323,79.2626


In [38]:
# ============================================================
# 29. FINAL RANDOM FOREST — CONSOLIDATED FOLD RESULTS
# ============================================================

rf_final_fold_results = []

for fold_year in [2023, 2024, 2025]:

    fold_predictions = (
        rf_final_predictions_df[
            rf_final_predictions_df["FOLD"] == fold_year
        ]
        .copy()
    )

    y_true = fold_predictions["ACTUAL_PEAK_RISK"]
    y_pred = fold_predictions["RF_PREDICTION"]
    y_prob = fold_predictions["RF_PROBABILITY"]

    metrics = evaluate_peak_risk_classifier(
        y_true,
        y_pred,
        y_prob
    )

    rf_final_fold_results.append({
        "FOLD": fold_year,
        "TEST_ROWS": len(fold_predictions),

        "ACTUAL_POSITIVE_RATE_PERCENT":
            y_true.mean() * 100,

        **metrics
    })


rf_final_fold_results = pd.DataFrame(
    rf_final_fold_results
)

display(
    rf_final_fold_results.round(4)
)

,FOLD,TEST_ROWS,ACTUAL_POSITIVE_RATE_PERCENT,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER
0,2023,420480,2.9909,0.3431,0.8849,0.4944,0.9163,7.7152,0.4628,0.9738,0.0420
1,2024,421632,4.0927,0.3767,0.9533,0.5400,0.9430,10.3578,0.6905,0.9816,0.0480
2,2025,419928,7.0586,0.5206,0.9388,0.6698,0.9366,12.7288,0.7902,0.9811,0.0456


In [39]:
# ============================================================
# 30. FINAL RANDOM FOREST — GLOBAL OUT-OF-SAMPLE RESULTS
# ============================================================

global_y_true = (
    rf_final_predictions_df["ACTUAL_PEAK_RISK"]
)

global_y_pred = (
    rf_final_predictions_df["RF_PREDICTION"]
)

global_y_prob = (
    rf_final_predictions_df["RF_PROBABILITY"]
)

global_metrics = evaluate_peak_risk_classifier(
    global_y_true,
    global_y_pred,
    global_y_prob
)

rf_final_global_results = pd.DataFrame([{
    "FOLD": "GLOBAL",

    "TEST_ROWS":
        len(rf_final_predictions_df),

    "ACTUAL_POSITIVE_RATE_PERCENT":
        global_y_true.mean() * 100,

    **global_metrics
}])

display(
    rf_final_global_results.round(4)
)

,FOLD,TEST_ROWS,ACTUAL_POSITIVE_RATE_PERCENT,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER
0,GLOBAL,1262040,4.7124,0.4276,0.9316,0.5862,0.9350,10.2663,0.6727,0.9790,0.0452


In [40]:
# ============================================================
# 31. FINAL RANDOM FOREST — TEAM RESULTS TABLE FORMAT
# ============================================================

rf_final_summary = pd.concat(
    [
        rf_final_global_results,
        rf_final_fold_results
    ],
    ignore_index=True
)

rf_final_summary.insert(
    0,
    "MODEL",
    "Random Forest Classifier"
)

rf_final_summary = rf_final_summary[
    [
        "MODEL",
        "FOLD",
        "PRECISION",
        "RECALL",
        "F1",
        "BALANCED_ACCURACY",
        "POSITIVE_RATE_PERCENT",
        "PR_AUC",
        "ROC_AUC",
        "BRIER"
    ]
]

display(
    rf_final_summary.round(4)
)

,MODEL,FOLD,PRECISION,RECALL,F1,BALANCED_ACCURACY,POSITIVE_RATE_PERCENT,PR_AUC,ROC_AUC,BRIER
0,Random Forest Classifier,GLOBAL,0.4276,0.9316,0.5862,0.9350,10.2663,0.6727,0.9790,0.0452
1,Random Forest Classifier,2023,0.3431,0.8849,0.4944,0.9163,7.7152,0.4628,0.9738,0.0420
2,Random Forest Classifier,2024,0.3767,0.9533,0.5400,0.9430,10.3578,0.6905,0.9816,0.0480
3,Random Forest Classifier,2025,0.5206,0.9388,0.6698,0.9366,12.7288,0.7902,0.9811,0.0456
